# miRNA Correlation Analysis

Correlation matrix for ~1800 miRNAs across ~1000 samples

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
miRNA = pd.read_csv('data/BRCA.miRNA_RPM_tumor.csv', sep=';')

In [4]:
# Set miRNA_ID as index
miRNA = miRNA.set_index('miRNA_ID')
print(f"Shape: {miRNA.shape[0]} miRNAs × {miRNA.shape[1]} samples")


Shape: 1881 miRNAs × 1078 samples


In [7]:
# Check for rows that are entirely zeros
zero_rows = (miRNA == 0).all(axis=1)
print(f"Number of rows (miRNAs) with all zeros: {zero_rows.sum()}")
if zero_rows.any():
    print("miRNA IDs with all zeros:")
    print(miRNA.index[zero_rows].tolist())


Number of rows (miRNAs) with all zeros: 283
miRNA IDs with all zeros:
['hsa-mir-103b-2', 'hsa-mir-1183', 'hsa-mir-1184-2', 'hsa-mir-1200', 'hsa-mir-1202', 'hsa-mir-1208', 'hsa-mir-1233-1', 'hsa-mir-1244-3', 'hsa-mir-1255b-1', 'hsa-mir-1255b-2', 'hsa-mir-1268a', 'hsa-mir-1268b', 'hsa-mir-1273d', 'hsa-mir-1273e', 'hsa-mir-1273f', 'hsa-mir-1273g', 'hsa-mir-1279', 'hsa-mir-1285-2', 'hsa-mir-1297', 'hsa-mir-1302-1', 'hsa-mir-1302-10', 'hsa-mir-1302-11', 'hsa-mir-1302-2', 'hsa-mir-1302-4', 'hsa-mir-1302-5', 'hsa-mir-1302-6', 'hsa-mir-1302-7', 'hsa-mir-1302-8', 'hsa-mir-1302-9', 'hsa-mir-1321', 'hsa-mir-1324', 'hsa-mir-1470', 'hsa-mir-1471', 'hsa-mir-1587', 'hsa-mir-1827', 'hsa-mir-1973', 'hsa-mir-2052', 'hsa-mir-2053', 'hsa-mir-2054', 'hsa-mir-2392', 'hsa-mir-2909', 'hsa-mir-297', 'hsa-mir-300', 'hsa-mir-302e', 'hsa-mir-302f', 'hsa-mir-3118-1', 'hsa-mir-3118-2', 'hsa-mir-3118-3', 'hsa-mir-3118-4', 'hsa-mir-3119-1', 'hsa-mir-3123', 'hsa-mir-3134', 'hsa-mir-3179-1', 'hsa-mir-3179-2', 'hsa-mir-

In [10]:
# Removing zeros from miRNA data
miRNA = miRNA[~zero_rows]
print(f"Shape after removing zeros: {miRNA.shape[0]} miRNAs × {miRNA.shape[1]} samples")

Shape after removing zeros: 1598 miRNAs × 1078 samples


/var/folders/pl/097pyytj0n59vsv43sybymdw0000gn/T/ipykernel_49053/619312852.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  miRNA = miRNA[~zero_rows]


In [5]:
# Compute correlation matrix between miRNAs (across samples)
# Log transform first to handle the skewed distribution
log_miRNA = np.log2(miRNA + 1)

print("Computing correlation matrix...")
corr_matrix = log_miRNA.T.corr()  # Transpose so we correlate miRNAs across samples
print(f"Correlation matrix shape: {corr_matrix.shape}")


Computing correlation matrix...
Correlation matrix shape: (1881, 1881)


## Exploring the RNAseq data

In [30]:
RNAseq = pd.read_csv('data/BRCA.RNA_seq_TPM.csv', sep=';')
RNAseq = RNAseq.set_index('gene_id')
RNAseq = RNAseq.T

# Check for all zero columns in RNAseq
all_zero_cols = (RNAseq == 0).all(axis=0)
if all_zero_cols.any():
    print("Columns with all zeros found:")
    print(len(RNAseq.columns[all_zero_cols].tolist()))
else:
    print("No all-zero columns found in RNAseq.")

# Remove all zero columns from RNAseq
RNAseq = RNAseq.loc[:, (RNAseq != 0).any(axis=0)]
RNAseq.shape

# Check for all zero rows in RNAseq
all_zero_rows = (RNAseq == 0).all(axis=1)

print("Shape of RNAseq after removing all zero columns:", RNAseq.shape)

Columns with all zeros found:
2670
Shape of RNAseq after removing all zero columns: (1095, 57990)


In [31]:
# Log transform the RNAseq data
RNAseq = np.log2(RNAseq + 1)


## Exploring the DNA methylation data